### Reading Marks

In [1]:
import re

In [5]:
def parse_student_grades(file_path):
    grades = {}
    current_exam = None

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    for line in lines:
        line = line.strip()

        if not line:
            continue

        lower = line.lower()
        if 'midterm 1' in lower:
            current_exam = 'Midterm 1'
            continue
        elif 'midterm 2' in lower:
            current_exam = 'Midterm 2'
            continue
        elif 'final' in lower:
            current_exam = 'Final'
            continue

        match = re.match(r"(S\d+)\s+(\d+)", line)
        if match and current_exam is not None:
            student_id = match.group(1)
            score = float(match.group(2))
            grades[(student_id, current_exam)] = score

    return grades

In [8]:
grades = parse_student_grades('../Data/StudentGrades.txt')
grades

{('S01', 'Midterm 1'): 78.0,
 ('S02', 'Midterm 1'): 82.0,
 ('S03', 'Midterm 1'): 77.0,
 ('S04', 'Midterm 1'): 75.0,
 ('S05', 'Midterm 1'): 67.0,
 ('S06', 'Midterm 1'): 71.0,
 ('S07', 'Midterm 1'): 64.0,
 ('S08', 'Midterm 1'): 92.0,
 ('S09', 'Midterm 1'): 80.0,
 ('S10', 'Midterm 1'): 89.0,
 ('S01', 'Midterm 2'): 82.0,
 ('S02', 'Midterm 2'): 85.0,
 ('S03', 'Midterm 2'): 90.0,
 ('S04', 'Midterm 2'): 77.0,
 ('S05', 'Midterm 2'): 77.0,
 ('S06', 'Midterm 2'): 64.0,
 ('S07', 'Midterm 2'): 33.0,
 ('S08', 'Midterm 2'): 88.0,
 ('S09', 'Midterm 2'): 39.0,
 ('S10', 'Midterm 2'): 64.0,
 ('S01', 'Final'): 182.0,
 ('S02', 'Final'): 180.0,
 ('S03', 'Final'): 188.0,
 ('S04', 'Final'): 149.0,
 ('S05', 'Final'): 157.0,
 ('S06', 'Final'): 175.0,
 ('S07', 'Final'): 110.0,
 ('S08', 'Final'): 184.0,
 ('S09', 'Final'): 126.0,
 ('S10', 'Final'): 116.0}

### Reading CSV

In [24]:
import pandas as pd
import numpy as np
import os

In [25]:
def load_signal_csv(file_path, value_col=None):
    df = pd.read_csv(file_path)

    if value_col is None:
        value_col = df.columns[-1]
    
    values = df[value_col].values.astype(np.float32)
    return values

### ACC -> magnitude

In [9]:
def acc_magnitude(df):
    cols = df.columns.tolist()
    if len(cols) >= 3:
        x = df[cols[0]].values
        y = df[cols[1]].values
        z = df[cols[2]].values

        mag = np.sqrt(x**2 + y**2 + z**2)
        return mag.astype(np.float32)
    
    else:
        return df.iloc[:, -1].values.astype(np.float32)

### windowing

In [10]:
def make_windows(signal, window_size):
    windows = []
    n = len(signal)
    for start in range(0, n, window_size):
        end = start + window_size
        chunk = signal[start:end]
        if len(chunk) < window_size:
            break
        windows.append(chunk)
    return windows

### Feature extraction from each window

In [11]:
def extract_basic_features(window):
    return [
        np.mean(window),
        np.std(window),
        np.min(window),
        np.max(window)
    ]

### create sequence feature for one exam

In [12]:
def build_exam_sequence_features(exam_folder, window_size=60):
    features_per_signal = []

    # HR
    hr_path = os.path.join(exam_folder, 'HR,csv')
    if os.path.exists(hr_path):
        hr = load_signal_csv(hr_path)
        hr_wins = make_windows(hr, window_size)
        hr_feats = [extract_basic_features(w) for w in hr_wins]
        features_per_signal.append(np.array(hr_feats))

    # EDA
    eda_path = os.path.join(exam_folder, 'EDA.csv')
    if os.path.exists(eda_path):
        eda = load_signal_csv(eda_path)
        eda_wins = make_windows(eda, window_size)
        eda_feats = [extract_basic_features(w) for w in eda_wins]
        features_per_signal.append(np.array(eda_feats))

    # TEMP
    temp_path = os.path.join(exam_folder, "TEMP.csv")
    if os.path.exists(temp_path):
        temp = load_signal_csv(temp_path)
        temp_wins = make_windows(temp, window_size)
        temp_feats = [extract_basic_features(w) for w in temp_wins]
        features_per_signal.append(np.array(temp_feats))

    # BVP
    bvp_path = os.path.join(exam_folder, "BVP.csv")
    if os.path.exists(bvp_path):
        bvp = load_signal_csv(bvp_path)
        bvp_wins = make_windows(bvp, window_size)
        bvp_feats = [extract_basic_features(w) for w in bvp_wins]
        features_per_signal.append(np.array(bvp_feats))

    if len(features_per_signal) == 0:
        return None
    
    min_length = min(arr.shape[0] for arr in features_per_signal)
    features_per_signal = [arr[:min_length] for arr in features_per_signal]

    sequence = np.concatenate(features_per_signal, axis=1) # (T, F)
    return sequence.astype(np.float32)

### Build the complete dataset

In [22]:
def build_dataset(data_root, grades_dict, window_size=60):
    X = []
    y = []
    meta = []

    for (student_id, exam_name), score in grades_dict.items():
        exam_folder = os.path.join(data_root, student_id, exam_name)

        if not os.path.exists(exam_folder):
            print(f"Missing folder: {exam_folder}")
            continue

        seq = build_exam_sequence_features(exam_folder, window_size=window_size)
        
        if seq is None:
            print(f"No signals found in: {exam_folder}")
            continue

        X.append(seq)
        y.append(score)
        meta.append((student_id, exam_name))

    return X, np.array(y, dtype=np.float32), meta

### padding for sequences

In [14]:
import torch
from torch.utils.data import Dataset

In [15]:
class StressDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets

    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, index):
        x = torch.tensor(self.sequences[index], dtype=torch.float32)
        y = torch.tensor(self.targets[index], dtype=torch.float32)
        return x, y
    
def collate_fn(batch):
    sequences, targets = zip(*batch)

    lengths = torch.tensor([seq.shape[0] for seq in sequences], dtype=torch.long)
    feature_dim = sequences[0].shape[1]
    max_len = max(lengths).item()

    padded = torch.zeros(len(sequences), max_len, feature_dim, dtype=torch.float32)

    for i, seq in enumerate(sequences):
        padded[i, :seq.shape[0], :] = seq

    targets = torch.stack(targets)
    return padded, lengths, targets

### Attention Modle

In [16]:
import torch.nn as nn
import torch

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, outputs, mask=None):
        # outputs: (B, T, H)
        scores = self.attn(outputs).squeeze(-1)  # (B, T)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = torch.softmax(scores, dim=1)  # (B, T)
        context = torch.sum(outputs * weights.unsqueeze(-1), dim=1)  # (B, H)

        return context, weights


In [17]:
class AttentionRegressionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, bidirectional=True):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )

        gru_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.attention = TemporalAttention(gru_output_dim)

        self.regressor = nn.Sequential(
            nn.Linear(gru_output_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x, lengths):
        # x: (B, T, F)

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_outputs, _ = self.gru(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)

        max_len = outputs.size(1)
        device = x.device

        mask = torch.arange(max_len, device=device).unsqueeze(0) < lengths.unsqueeze(1)

        context, attn_weights = self.attention(outputs, mask)
        pred = self.regressor(context).squeeze(-1)

        return pred, attn_weights


### Training

In [18]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn.functional as F

def train_model(model, train_loader, val_loader, device, epochs=50, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for x, lengths, y in train_loader:
            x = x.to(device)
            lengths = lengths.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            pred, _ = model(x, lengths)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        mae = 0.0

        with torch.no_grad():
            for x, lengths, y in val_loader:
                x = x.to(device)
                lengths = lengths.to(device)
                y = y.to(device)

                pred, _ = model(x, lengths)
                loss = criterion(pred, y)

                val_loss += loss.item()
                mae += torch.mean(torch.abs(pred - y)).item()

        val_loss /= len(val_loader)
        mae /= len(val_loader)

        print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val MAE: {mae:.4f}")


### Split Data

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

grades = parse_student_grades("../Data/StudentGrades.txt")
X, y, meta = build_dataset("../Data", grades, window_size=60)

In [33]:
from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    list(range(len(X))), test_size=0.2, random_state=42
)

X_train = [X[i] for i in train_idx]
y_train = y[train_idx]

X_val = [X[i] for i in val_idx]
y_val = y[val_idx]

train_dataset = StressDataset(X_train, y_train)
val_dataset = StressDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)


In [41]:
input_dim = X[0].shape[1]
model = AttentionRegressionModel(input_dim=input_dim, hidden_dim=64)

train_model(model, train_loader, val_loader, device, epochs=100, lr=1e-3)


Epoch 001 | Train Loss: 11735.3346 | Val Loss: 11791.5029 | Val MAE: 101.5387
Epoch 002 | Train Loss: 11466.5293 | Val Loss: 11483.2598 | Val MAE: 100.0083
Epoch 003 | Train Loss: 11131.6159 | Val Loss: 11109.2124 | Val MAE: 98.1123
Epoch 004 | Train Loss: 10736.3869 | Val Loss: 10667.0278 | Val MAE: 95.8239
Epoch 005 | Train Loss: 10285.3248 | Val Loss: 10152.1360 | Val MAE: 93.0991
Epoch 006 | Train Loss: 9758.5518 | Val Loss: 9556.7822 | Val MAE: 89.8605
Epoch 007 | Train Loss: 9149.4053 | Val Loss: 8904.9382 | Val MAE: 86.1657
Epoch 008 | Train Loss: 8517.3355 | Val Loss: 8197.3398 | Val MAE: 81.9618
Epoch 009 | Train Loss: 7824.3589 | Val Loss: 7457.1575 | Val MAE: 77.3209
Epoch 010 | Train Loss: 7091.9661 | Val Loss: 6716.2561 | Val MAE: 72.3778
Epoch 011 | Train Loss: 6399.0408 | Val Loss: 5947.5940 | Val MAE: 66.8303
Epoch 012 | Train Loss: 5643.4207 | Val Loss: 5224.6255 | Val MAE: 61.1658
Epoch 013 | Train Loss: 5044.2950 | Val Loss: 4520.3560 | Val MAE: 55.1117
Epoch 014 | T

### Attention analysis

In [35]:
def inspect_attention(model, dataset, idx, device):
    model.eval()
    x, y = dataset[idx]
    lengths = torch.tensor([x.shape[0]], dtype=torch.long)

    with torch.no_grad():
        pred, attn = model(x.unsqueeze(0).to(device), lengths.to(device))

    print("True score:", y.item())
    print("Predicted score:", pred.item())
    print("Attention weights:", attn.squeeze(0).cpu().numpy())


In [39]:
inspect_attention(model, train_dataset, 3, device)

True score: 78.0
Predicted score: 97.34552001953125
Attention weights: [0.00024193 0.00040896 0.00060743 0.00088156 0.00107977 0.00118531
 0.00124104 0.00127235 0.00129078 0.00130404 0.00131363 0.00132006
 0.00132509 0.00132887 0.00133168 0.00133385 0.00133561 0.00133669
 0.0013377  0.00133864 0.00133916 0.00133967 0.00133988 0.00134028
 0.00134048 0.00134086 0.00134095 0.00134111 0.00134109 0.00134116
 0.00134148 0.00134147 0.00134152 0.00134151 0.00134171 0.00134173
 0.00134181 0.00134173 0.00134175 0.00134167 0.00134181 0.0013419
 0.0013419  0.001342   0.00134203 0.00134221 0.00134221 0.00134218
 0.00134213 0.00134217 0.00134233 0.00134237 0.00134227 0.00134232
 0.00134245 0.00134251 0.00134257 0.0013426  0.00134255 0.00134265
 0.00134258 0.00134268 0.00134272 0.00134267 0.0013427  0.00134285
 0.0013429  0.00134285 0.00134296 0.00134299 0.00134294 0.00134307
 0.00134305 0.00134312 0.00134326 0.00134308 0.00134328 0.00134328
 0.00134321 0.00134337 0.00134341 0.0013433  0.00134346 0.0